# Notebook 6 — Anomaly Detection: Model Training

Trains both Task 2 models — Isolation Forest (unsupervised) and Random Forest (supervised) — with training-time instrumentation matching Notebooks 4/5. Every step below has been run against the real dataset and verified before being included here.

## Step 0 — Imports & Reproducibility

In [ ]:
import os, sys, random, time
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier

SEED = 42
random.seed(SEED); os.environ['PYTHONHASHSEED'] = str(SEED); np.random.seed(SEED)
print("pandas:", pd.__version__)

## Step 0b — Fix Working Directory

In [ ]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())
os.makedirs("models", exist_ok=True)

## Step 1 — Load

In [ ]:
INPUT_PATH = "data/processed/master_features.csv"
assert os.path.exists(INPUT_PATH), f"Missing input: {INPUT_PATH} — run 02_feature_engineering.ipynb first"

df = pd.read_csv(INPUT_PATH, parse_dates=['timestamp']).set_index('timestamp')
print(df.shape)

## Step 2 — Feature Selection & Stratified Split

**Deliberately includes** `it_load_kwh`, `cooling_load_kwh`, `support_load_kwh`, `pue` — unlike Notebook 4, where these were excluded as arithmetic leakage. Here, `anomaly_flag` is a *rule computed from thresholds on* these columns, not their algebraic sum — including them is exactly what a real fault-detection system would do, reading the same operational sensors.

In [ ]:
feature_cols = ['ambient_temp_c','relative_humidity_pct','chilled_water_temp_c',
                 'server_inlet_temp_c','under_floor_pressure_pa','it_load_kwh',
                 'cooling_load_kwh','support_load_kwh','total_power_kwh','pue',
                 'hour_sin','hour_cos','month_sin','month_cos','is_weekend']

X = df[feature_cols]
y = (df['anomaly_flag'] == 'failure').astype(int)
print(f"Total failures: {y.sum()} ({y.mean()*100:.2f}%)")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f"Train: {len(X_train)} rows, {y_train.sum()} failures ({y_train.mean()*100:.2f}%)")
print(f"Test:  {len(X_test)} rows, {y_test.sum()} failures ({y_test.mean()*100:.2f}%)")
assert abs(y_train.mean() - y_test.mean()) < 0.005, "Stratified split failed to preserve ratio"

## Step 3 — Scale (fit on train only)

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
print("Scaled shapes:", X_train_sc.shape, X_test_sc.shape)

## Step 4 — Train Model 1: Isolation Forest (unsupervised)

**Contamination sourced from `y_train.mean()` only** — not the full dataset — so this hyperparameter never sees test-set information. The model itself never sees `y_train` during `.fit()` either; contamination only tells it what *fraction* to expect, not which rows.

In [ ]:
contamination_rate = y_train.mean()
print(f"Contamination rate (from y_train only): {contamination_rate:.4f}")

t0 = time.perf_counter()
iso_forest = IsolationForest(n_estimators=200, contamination=contamination_rate, random_state=SEED)
iso_forest.fit(X_train_sc)
train_time_iso = time.perf_counter() - t0
print(f"Trained in {train_time_iso:.2f}s")

## Step 5 — Train Model 2: Random Forest (supervised)

**Expect a very high score here — this is legitimate, not a bug.** `anomaly_flag`'s rule uses several of these exact columns as thresholds, so the classifier has direct access to the rule's own deciding variables. It's closer to recovering a known rule than detecting genuinely novel anomalies — worth stating plainly rather than presenting the number without context.

In [ ]:
t0 = time.perf_counter()
rf_clf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced',
                                 random_state=SEED, n_jobs=-1)
rf_clf.fit(X_train_sc, y_train)
train_time_rf = time.perf_counter() - t0
print(f"Trained in {train_time_rf:.2f}s")

## Step 6 — Save Models, Scaler & Test Data

Includes `train_times` and `feature_cols` in the bundle — Notebook 7 needs both, and this keeps the same structure as Notebook 4's handoff to Notebook 5.

In [ ]:
joblib.dump(iso_forest, 'models/anomaly_isoforest.pkl')
joblib.dump(rf_clf, 'models/anomaly_rf.pkl')
joblib.dump(scaler, 'models/anomaly_scaler.pkl')
joblib.dump({
    'X_test': X_test, 'X_test_sc': X_test_sc, 'y_test': y_test,
    'feature_cols': feature_cols,
    'train_times': {'IsolationForest': train_time_iso, 'RandomForest': train_time_rf}
}, 'models/anomaly_test_data.pkl')

for f in ['anomaly_isoforest.pkl','anomaly_rf.pkl','anomaly_scaler.pkl','anomaly_test_data.pkl']:
    assert os.path.exists(f'models/{f}'), f"Failed to save {f}"
print("Saved 3 model/scaler files + test data bundle. Ready for Notebook 7.")